# Tahap 01: Data Scraping
Tahap ini bertujuan untuk mengumpulkan data komentar publik dari media sosial (Instagram) menggunakan pustaka `instagrapi`. Proses ini mencakup autentikasi sesi, ekstraksi *shortcode* dari URL *post*, dan pengambilan komentar untuk kemudian disimpan dalam format *dataframe*.

## 1.1 Import Library dan Autentikasi Pengguna
Inisialisasi *client* dan pemuatan sesi untuk menghindari verifikasi berulang saat melakukan *login*.

In [ ]:
# --- LIBRARIES ---
import os
import json
from pathlib import Path
from getpass import getpass
from dotenv import load_dotenv
from instagrapi import Client

# Memuat environment variables (opsional)
load_dotenv()  

# Konfigurasi file sesi
SESSION_FILE = Path("insta_session.json")
cl = Client()

# Memuat sesi sebelumnya jika tersedia untuk menghindari verifikasi berulang
if SESSION_FILE.exists():
    try:
        cl.load_settings(SESSION_FILE)
    except Exception:
        pass  # Lanjut ke login reguler jika gagal memuat sesi

# Kredensial pengguna
USERNAME = os.getenv("IG_USERNAME") or input("Masukkan IG username: ")
PASSWORD = os.getenv("IG_PASSWORD") or getpass("Masukkan IG password: ")

# Eksekusi login
cl.login(USERNAME, PASSWORD)

# Menyimpan sesi untuk proses login selanjutnya
try:
    cl.dump_settings(SESSION_FILE)
except Exception:
    pass

## 1.2 Fungsi Ekstraksi dan Pemrosesan Komentar
Mendefinisikan fungsi-fungsi modular untuk memproses URL, mengekstrak informasi relevan dari setiap komentar, dan merangkumnya ke dalam struktur baris data.

In [ ]:
import pandas as pd
from tqdm import tqdm
from urllib.parse import urlparse
import re
from datetime import timezone

def extract_shortcode_from_url(url: str) -> str:
    """
    Menerima URL post (format /p/SHORTCODE/ atau /reel/SHORTCODE/)
    dan mengekstrak nilai shortcode.
    """
    path = urlparse(url).path
    parts = [p for p in path.split('/') if p]
    
    if len(parts) >= 2 and parts[0] in {"p", "reel", "tv"}:
        return parts[1]
    
    # Fallback: menggunakan reguler expression umum
    m = re.search(r"/(p|reel|tv)/([^/?#]+)/?", path)
    if m:
        return m.group(2)
        
    raise ValueError(f"Gagal menemukan shortcode dari URL: {url}")

def comments_to_rows(comments, post_url: str, shortcode: str) -> list:
    """
    Memproses objek komentar mentah menjadi format baris (dictionary) 
    untuk kebutuhan DataFrame.
    """
    rows = []
    for c in comments:
        cid = getattr(c, "pk", None) or getattr(c, "id", None)
        created = getattr(c, "created_at_utc", None)
        
        if created and created.tzinfo is None:
            created = created.replace(tzinfo=timezone.utc)

        username = None
        if getattr(c, "user", None) is not None:
            username = getattr(c.user, "username", None)

        text = getattr(c, "text", None)
        like_count = getattr(c, "like_count", None)
        comment_url = f"{post_url.split('?')[0]}?comment_id={cid}" if cid else None

        rows.append({
            "id": str(cid) if cid is not None else None,
            "timestamp": created.isoformat() if created else None,
            "ownerUsername": username,
            "text": text,
            "likesCount": like_count,
            "postUrl": post_url,
            "commentUrl": comment_url
        })
    return rows

def fetch_comments_for_post_url(client: Client, post_url: str, amount: int = 0) -> pd.DataFrame:
    """
    Mengambil seluruh komentar dari URL yang diberikan.
    Parameter amount=0 akan mengambil semua komentar yang tersedia.
    """
    shortcode = extract_shortcode_from_url(post_url)
    media_pk = client.media_pk_from_url(post_url)
    
    comments = client.media_comments(media_pk, amount=amount)
    rows = comments_to_rows(comments, post_url, shortcode)
    
    return pd.DataFrame(rows)

## 1.3 Eksekusi Scraping dan Penyimpanan Data
Melakukan iterasi pada daftar URL target, mengumpulkan data komentar, dan menggabungkannya menjadi satu DataFrame tunggal.

In [ ]:
# Daftar URL post layanan publik digital atau target lainnya yang ingin di-scrap
post_urls = [ 
    # "https://www.instagram.com/p/XXXXXXXXXXX/",
    # "https://www.instagram.com/reel/YYYYYYYYYYY/",
]

all_dfs = []

# Proses iterasi pengambilan komentar
for url in tqdm(post_urls):
    try:
        df_one = fetch_comments_for_post_url(cl, url, amount=0)
        all_dfs.append(df_one)
    except Exception as e:
        print(f"Gagal mengambil data dari {url}: {e}")

# Penggabungan hasil ke dalam satu DataFrame utama
if all_dfs:
    df = pd.concat(all_dfs, ignore_index=True)
else:
    df = pd.DataFrame(columns=[
        "id", "timestamp", "ownerUsername", "text", 
        "likesCount", "postUrl", "commentUrl"
    ])

# Menampilkan 5 baris pertama untuk validasi
df.head()